# Network Analysis — AI Coding Assistant Trust & Distrust
**COSC 3047 Assignment 2**

This notebook constructs and analyses user interaction networks from public discussion about AI coding assistants.

**Network definitions:**
- **Directed reply graph** — users connected by direct reply edges. This is used for PageRank, in-degree, and betweenness centrality.
- **Undirected community graph** — users connected by reply edges or by co-participating in the same discussion thread. This is used for Louvain community detection.

GitHub issue comments are not direct replies, so they are not used in the directed reply graph. Instead, GitHub user-issue participation edges are projected into user-user co-participation edges for community detection.

**Analyses performed:**
1. Directed reply graph construction from YouTube and Hacker News reply edges
2. Basic directed graph statistics
3. Centrality analysis (PageRank, in-degree, betweenness) on direct replies
4. Community detection (Louvain) on reply plus co-participation edges
5. Cross-analysis linking network position to sentiment and topic labels

**Input:** `data/processed/combined/combined_edges.json`, `data/processed/nlp/comments_with_topics.csv`
**Output:** `data/processed/network/`


## 1. Setup & Imports

In [ ]:
import json
import logging
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import seaborn as sns
from community import community_louvain

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

In [ ]:
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "data/processed/network"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EDGES_PATH = PROJECT_ROOT / "data/processed/combined/combined_edges.json"
NLP_PATH = PROJECT_ROOT / "data/processed/nlp/comments_with_topics.csv"
COMBINED_COMMENTS_PATH = PROJECT_ROOT / "data/processed/combined/combined_comments.csv"

print(f"Project root:       {PROJECT_ROOT}")
print(f"Edges path:         {EDGES_PATH.exists()} ({EDGES_PATH})")
print(f"NLP data:           {NLP_PATH.exists()} ({NLP_PATH})")
print(f"Combined comments:  {COMBINED_COMMENTS_PATH.exists()} ({COMBINED_COMMENTS_PATH})")


## 2. Colour Maps

In [ ]:
SENTIMENT_COLORS = {
    "positive": "#4CAF50",
    "neutral":  "#FF9800",
    "negative": "#F44336",
    "unknown":  "#9E9E9E"
}

TOPIC_COLORS = {
    "productivity":    "#2196F3",
    "reliability":     "#F44336",
    "security":        "#FF9800",
    "cost":            "#9C27B0",
    "code_ownership":  "#009688",
    "job_displacement":"#795548",
    "other":           "#9E9E9E",
    "unknown":         "#BDBDBD"
}

print("Colour maps defined.")

## 3. Load Edge Data

The combined edge file contains four types of edges:
- `reply` (YouTube + HN) — directed user→user reply relationships
- `user_thread` (HN + GitHub) — bipartite user→thread participation

For the **directed graph** (PageRank, centrality) we use `reply` edges only.
For **community detection** (Louvain) we use all edges for richer structure.
Issue key nodes (containing `#`) are filtered out — they are thread IDs not users.

In [ ]:
with open(EDGES_PATH, encoding="utf-8") as f:
    raw_edges = json.load(f)

edges_df = pd.DataFrame(raw_edges)
log.info(f"Loaded {len(edges_df):,} total edges")

required_columns = {"source", "target", "platform", "edge_type", "weight"}
missing_columns = required_columns - set(edges_df.columns)
if missing_columns:
    raise ValueError(f"Missing required edge columns: {sorted(missing_columns)}")

edges_df["source"] = edges_df["source"].fillna("").astype(str).str.strip()
edges_df["target"] = edges_df["target"].fillna("").astype(str).str.strip()
edges_df["weight"] = pd.to_numeric(edges_df["weight"], errors="coerce").fillna(1).astype(int)

# Remove unusable rows only. Thread IDs are retained for projection below.
edges_df = edges_df[(edges_df["source"].str.len() > 0) & (edges_df["target"].str.len() > 0)].copy()
edges_df = edges_df[edges_df["source"] != edges_df["target"]].copy()

reply_edges_df = edges_df[edges_df["edge_type"] == "reply"].copy()
user_thread_df = edges_df[edges_df["edge_type"] == "user_thread"].copy()

# Directed graph: direct user replies only.
reply_df = (
    reply_edges_df
    .groupby(["source", "target", "platform"], as_index=False)
    .agg(weight=("weight", "sum"))
)

# Community graph: direct replies plus user-user co-participation in the same thread.
co_participation_rows = []
for (platform, thread_id), group in user_thread_df.groupby(["platform", "target"]):
    users = sorted(set(group["source"].dropna().astype(str)))
    if len(users) < 2:
        continue
    for index, source in enumerate(users):
        for target in users[index + 1:]:
            co_participation_rows.append({
                "source": source,
                "target": target,
                "platform": platform,
                "edge_type": "co_thread",
                "weight": 1,
            })

co_thread_df = pd.DataFrame(co_participation_rows)
if len(co_thread_df):
    co_thread_df = (
        co_thread_df
        .groupby(["source", "target", "platform", "edge_type"], as_index=False)
        .agg(weight=("weight", "sum"))
    )

community_edges_df = pd.concat([
    reply_df.assign(edge_type="reply"),
    co_thread_df,
], ignore_index=True)

print(f"Total raw edges:                 {len(raw_edges):,}")
print(f"Edges after basic cleaning:      {len(edges_df):,}")
print(f"Reply edges for directed graph:  {len(reply_df):,}")
print(f"User-thread edges for projection:{len(user_thread_df):,}")
print(f"Projected co-thread edges:       {len(co_thread_df):,}")
print(f"Community graph edge rows:       {len(community_edges_df):,}")
print()
print("Raw edge types:")
print(edges_df.groupby(["platform", "edge_type"]).size().to_string())
print()
print("Community edge rows:")
print(community_edges_df.groupby(["platform", "edge_type"]).size().to_string())
reply_df.head(3)


## 4. Load NLP Data

Sentiment scores and topic labels are loaded and aggregated per user.
These are attached as node attributes to enable cross-analysis with network position.

In [ ]:
if NLP_PATH.exists():
    nlp_df = pd.read_csv(NLP_PATH)
    print(f"Loaded NLP data: {NLP_PATH}")
elif COMBINED_COMMENTS_PATH.exists():
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

    nlp_df = pd.read_csv(COMBINED_COMMENTS_PATH)
    print(f"NLP file missing, using combined comments: {COMBINED_COMMENTS_PATH}")
    print("Computing VADER sentiment and simple theme labels for network profiling.")

    analyser = SentimentIntensityAnalyzer()
    sentiment_scores = nlp_df["text"].fillna("").astype(str).apply(analyser.polarity_scores)
    nlp_df["sentiment_compound"] = sentiment_scores.apply(lambda scores: scores["compound"])
    nlp_df["sentiment_label"] = nlp_df["sentiment_compound"].apply(
        lambda score: "positive" if score >= 0.05 else "negative" if score <= -0.05 else "neutral"
    )

    theme_keywords = {
        "productivity": ["productiv", "workflow", "faster", "time", "automate", "agent", "assistant", "build", "write", "code"],
        "reliability": ["bug", "error", "crash", "fail", "broken", "issue", "problem", "wrong", "hallucinat", "fix"],
        "security": ["security", "privacy", "private", "data", "leak", "permission", "safe", "risk"],
        "cost": ["cost", "price", "pricing", "subscription", "expensive", "token", "limit", "rate"],
        "code_ownership": ["license", "copyright", "ownership", "open source", "source", "training data"],
        "job_displacement": ["replace", "job", "career", "junior", "developer", "programmer", "engineer", "layoff"],
    }

    def assign_theme(text):
        text = str(text).lower()
        scores = {
            theme: sum(1 for keyword in keywords if keyword in text)
            for theme, keywords in theme_keywords.items()
        }
        best_theme, best_score = max(scores.items(), key=lambda item: item[1])
        return best_theme if best_score > 0 else "other"

    text_for_topics = nlp_df["text_clean"].fillna(nlp_df["text"].fillna(""))
    nlp_df["topic_label"] = text_for_topics.apply(assign_theme)
else:
    raise FileNotFoundError(
        "Missing both comments_with_topics.csv and combined_comments.csv. "
        "Run data_merge.ipynb first, then rerun this notebook."
    )

# Normalise column names across platform-specific or combined NLP outputs.
if "sentiment_compound" not in nlp_df.columns and "sentimentCompound" in nlp_df.columns:
    nlp_df = nlp_df.rename(columns={"sentimentCompound": "sentiment_compound"})
if "sentiment_label" not in nlp_df.columns and "sentimentLabel" in nlp_df.columns:
    nlp_df = nlp_df.rename(columns={"sentimentLabel": "sentiment_label"})
if "topic_label" not in nlp_df.columns and "topicLabel" in nlp_df.columns:
    nlp_df = nlp_df.rename(columns={"topicLabel": "topic_label"})

# Map column names for YouTube-only, platform-specific, and combined outputs.
author_col = "commentAuthor" if "commentAuthor" in nlp_df.columns else "author"
text_col = "commentText" if "commentText" in nlp_df.columns else "text" if "text" in nlp_df.columns else "text_clean"
likes_col = "commentLikeCount" if "commentLikeCount" in nlp_df.columns else "like_count"

if likes_col not in nlp_df.columns:
    nlp_df[likes_col] = 0
if "topic_label" not in nlp_df.columns:
    nlp_df["topic_label"] = "unknown"

nlp_df = nlp_df[nlp_df[author_col].notna()].copy()
nlp_df[author_col] = nlp_df[author_col].astype(str)

user_profiles = (
    nlp_df.groupby(author_col)
    .agg(
        avg_sentiment=("sentiment_compound", "mean"),
        comment_count=(text_col, "count"),
        dominant_topic=("topic_label", lambda x: x.value_counts().index[0] if len(x) > 0 else "unknown"),
        total_likes=(likes_col, "sum")
    )
    .reset_index()
    .rename(columns={author_col: "author"})
)
user_profiles["sentiment_label"] = user_profiles["avg_sentiment"].apply(
    lambda x: "positive" if x >= 0.05 else "negative" if x <= -0.05 else "neutral"
)

user_dict = user_profiles.set_index("author").to_dict(orient="index")

print(f"Rows used for network NLP profiles: {len(nlp_df):,}")
print(f"User profiles: {len(user_profiles):,} unique users")
print("Sentiment distribution:")
print(user_profiles["sentiment_label"].value_counts().to_string())
print("Dominant topic distribution:")
print(user_profiles["dominant_topic"].value_counts().head(10).to_string())
user_profiles.head(3)


## 5. Graph Construction

A directed weighted graph is built from reply edges only.
- **Directed** — a reply from A to B does not imply B replied to A
- **Weighted** — edge weight = number of times A replied to B

NLP attributes are attached to each node for cross-analysis.

In [ ]:
G = nx.DiGraph()

for _, row in reply_df.iterrows():
    G.add_edge(row["source"], row["target"],
               weight=row["weight"], platform=row["platform"])

# Attach NLP attributes to nodes
for node in G.nodes():
    if node in user_dict:
        for attr, val in user_dict[node].items():
            G.nodes[node][attr] = val
    else:
        G.nodes[node]["avg_sentiment"]  = 0.0
        G.nodes[node]["sentiment_label"] = "unknown"
        G.nodes[node]["dominant_topic"]  = "unknown"
        G.nodes[node]["comment_count"]   = 0
        G.nodes[node]["total_likes"]     = 0

print(f"Graph constructed:")
print(f"  Nodes (users):    {G.number_of_nodes():,}")
print(f"  Edges (replies):  {G.number_of_edges():,}")
print(f"  Density:          {nx.density(G):.6f}")
print(f"  Is DAG:           {nx.is_directed_acyclic_graph(G)}")

## 6. Basic Network Statistics

Degree distribution reveals whether the network follows a power law — typical of social networks where a few users attract most replies.

In [ ]:
G_undirected = G.to_undirected()
components   = list(nx.connected_components(G_undirected))
largest_cc   = max(components, key=len)

in_degrees  = dict(G.in_degree())
out_degrees = dict(G.out_degree())

print(f"Connected components:       {len(components):,}")
print(f"Largest component:          {len(largest_cc):,} nodes ({len(largest_cc)/G.number_of_nodes()*100:.1f}%)")
print(f"Isolated nodes:             {sum(1 for c in components if len(c) == 1):,}")
print()
print(f"In-degree  — max: {max(in_degrees.values())}, mean: {np.mean(list(in_degrees.values())):.2f}")
print(f"Out-degree — max: {max(out_degrees.values())}, mean: {np.mean(list(out_degrees.values())):.2f}")
print()
print("Top 5 most replied-to users:")
for user, deg in sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:5]:
    s = G.nodes[user].get("sentiment_label", "unknown")
    t = G.nodes[user].get("dominant_topic", "unknown")
    print(f"  {user[:40]:40s} in-degree: {deg:4d} | {s} | {t}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Network Degree Distribution", fontsize=13, fontweight="bold")

in_deg_vals  = [d for _, d in G.in_degree()  if d > 0]
out_deg_vals = [d for _, d in G.out_degree() if d > 0]

axes[0].hist(in_deg_vals,  bins=50, color="#2196F3", edgecolor="white", alpha=0.85)
axes[0].set_title("In-Degree Distribution\n(replies received per user)")
axes[0].set_xlabel("In-Degree")
axes[0].set_ylabel("Number of Users")
axes[0].set_yscale("log")

axes[1].hist(out_deg_vals, bins=50, color="#FF9800", edgecolor="white", alpha=0.85)
axes[1].set_title("Out-Degree Distribution\n(replies made per user)")
axes[1].set_xlabel("Out-Degree")
axes[1].set_ylabel("Number of Users")
axes[1].set_yscale("log")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "degree_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved degree_distribution.png")

## 7. Centrality Analysis

Three centrality measures identify influential users:
- **PageRank** — overall influence accounting for quality of connections
- **In-degree centrality** — proportion of users who directly reply to this user
- **Betweenness centrality** — how often a user bridges two other users

Cross-referencing with sentiment and topic directly answers the research question: *do influential users shape trust or distrust?*

In [ ]:
log.info("Computing PageRank...")
pagerank = nx.pagerank(G, weight="weight", alpha=0.85)

log.info("Computing in-degree centrality...")
in_degree_centrality = nx.in_degree_centrality(G)

log.info("Computing betweenness centrality (largest component)...")
G_lcc        = G.subgraph(largest_cc).copy()
betweenness  = nx.betweenness_centrality(G_lcc, weight="weight", normalized=True)

centrality_df = pd.DataFrame({
    "user":                 list(pagerank.keys()),
    "pagerank":             list(pagerank.values()),
    "in_degree_centrality": [in_degree_centrality.get(u, 0) for u in pagerank],
    "betweenness":          [betweenness.get(u, 0)           for u in pagerank],
    "in_degree":            [in_degrees.get(u, 0)            for u in pagerank],
    "out_degree":           [out_degrees.get(u, 0)           for u in pagerank],
    "sentiment_label":      [G.nodes[u].get("sentiment_label", "unknown") for u in pagerank],
    "dominant_topic":       [G.nodes[u].get("dominant_topic",  "unknown") for u in pagerank],
    "avg_sentiment":        [G.nodes[u].get("avg_sentiment",   0.0)       for u in pagerank],
    "total_likes":          [G.nodes[u].get("total_likes",     0)         for u in pagerank],
})

centrality_df = centrality_df.sort_values("pagerank", ascending=False).reset_index(drop=True)
centrality_df.to_csv(OUTPUT_DIR / "centrality_scores.csv", index=False)

print("Top 15 most influential users (PageRank):")
print(centrality_df[["user", "pagerank", "in_degree", "sentiment_label", "dominant_topic"]].head(15).to_string(index=False))

### Centrality Visualisations

In [ ]:
top20 = centrality_df.head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Top 20 Most Influential Users — Centrality Analysis", fontsize=13, fontweight="bold")

# Plot 1 — PageRank coloured by sentiment
bar_colors = [SENTIMENT_COLORS.get(s, "#9E9E9E") for s in top20["sentiment_label"]]
axes[0].barh(top20["user"].str[:30][::-1], top20["pagerank"][::-1],
             color=bar_colors[::-1], edgecolor="white")
axes[0].set_title("PageRank Score\n(coloured by sentiment)")
axes[0].set_xlabel("PageRank")
legend_elements = [Patch(facecolor=v, label=k) for k, v in SENTIMENT_COLORS.items()]
axes[0].legend(handles=legend_elements, title="Sentiment", loc="lower right")

# Plot 2 — PageRank coloured by topic
bar_colors2 = [TOPIC_COLORS.get(t, "#9E9E9E") for t in top20["dominant_topic"]]
axes[1].barh(top20["user"].str[:30][::-1], top20["pagerank"][::-1],
             color=bar_colors2[::-1], edgecolor="white")
axes[1].set_title("PageRank Score\n(coloured by dominant topic)")
axes[1].set_xlabel("PageRank")
legend_elements2 = [Patch(facecolor=v, label=k) for k, v in TOPIC_COLORS.items() if k not in ["other", "unknown"]]
axes[1].legend(handles=legend_elements2, title="Topic", loc="lower right", fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "top_users_centrality.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved top_users_centrality.png")

## 8. Community Detection (Louvain)

Louvain maximises modularity to find communities of users who interact more with each other than with the rest of the network.

Community detection runs on an **undirected graph using all edge types** (reply + user_thread) — this gives richer community structure than reply edges alone, incorporating HN and GitHub participation patterns.

Each community is profiled by dominant sentiment and topic to reveal whether clusters form around trust or distrust.

In [ ]:
# Build undirected graph from reply and co-participation edges for Louvain.
G_community = nx.Graph()
for _, row in community_edges_df.iterrows():
    if G_community.has_edge(row["source"], row["target"]):
        G_community[row["source"]][row["target"]]["weight"] += row.get("weight", 1)
    else:
        G_community.add_edge(
            row["source"],
            row["target"],
            weight=row.get("weight", 1),
            platform=row.get("platform", "unknown"),
            edge_type=row.get("edge_type", "unknown"),
        )

log.info("Running Louvain community detection...")
partition = community_louvain.best_partition(G_community, weight="weight", random_state=42)

# Attach community labels to directed reply graph nodes where those users are present.
nx.set_node_attributes(G, {node: partition[node] for node in G.nodes() if node in partition}, "community")

num_communities = len(set(partition.values()))
modularity = community_louvain.modularity(partition, G_community)
print(f"Community graph nodes: {G_community.number_of_nodes():,}")
print(f"Community graph edges: {G_community.number_of_edges():,}")
print(f"Communities detected:  {num_communities:,}")
print(f"Modularity:            {modularity:.4f}")
print()

# Build community profiles from users with NLP attributes.
community_data = []
for comm_id in sorted(set(partition.values())):
    members = [u for u, c in partition.items() if c == comm_id]
    sentiments = []
    topics_list = []
    sentiment_scores = []

    for user in members:
        profile = user_dict.get(user)
        if profile and profile.get("sentiment_label", "unknown") != "unknown":
            sentiments.append(profile["sentiment_label"])
            topics_list.append(profile.get("dominant_topic", "unknown"))
            sentiment_scores.append(profile.get("avg_sentiment", 0))

    avg_sent = np.mean(sentiment_scores) if sentiment_scores else 0.0
    dominant_sentiment = Counter(sentiments).most_common(1)[0][0] if sentiments else "unknown"
    dominant_topic = Counter(topics_list).most_common(1)[0][0] if topics_list else "unknown"

    community_data.append({
        "community_id": comm_id,
        "size": len(members),
        "profiled_users": len(sentiments),
        "dominant_sentiment": dominant_sentiment,
        "dominant_topic": dominant_topic,
        "avg_sentiment_score": round(float(avg_sent), 4),
        "positive_pct": round(sentiments.count("positive") / len(sentiments) * 100, 1) if sentiments else 0,
        "negative_pct": round(sentiments.count("negative") / len(sentiments) * 100, 1) if sentiments else 0,
    })

community_df = pd.DataFrame(community_data).sort_values("size", ascending=False)
community_df.to_csv(OUTPUT_DIR / "community_profiles.csv", index=False)

print("Community profiles (top 10 by size):")
print(community_df.head(10).to_string(index=False))


### Community Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Community Structure — AI Coding Assistant Discussion Network", fontsize=13, fontweight="bold")

all_top = community_df.head(15)
known   = community_df[community_df["dominant_sentiment"] != "unknown"].head(15).sort_values("negative_pct", ascending=False)

# Plot 1 — Top 15 community sizes coloured by dominant sentiment
bar_colors = [SENTIMENT_COLORS.get(s, "#9E9E9E") for s in all_top["dominant_sentiment"]]
axes[0].bar(range(len(all_top)), all_top["size"], color=bar_colors, edgecolor="white")
axes[0].set_xticks(range(len(all_top)))
axes[0].set_xticklabels([f"C{cid}" for cid in all_top["community_id"]], rotation=45, fontsize=8)
axes[0].set_title("Top 15 Community Sizes\n(coloured by dominant sentiment)")
axes[0].set_xlabel("Community")
axes[0].set_ylabel("Number of Users")
legend_elements = [Patch(facecolor=v, label=k) for k, v in SENTIMENT_COLORS.items()]
axes[0].legend(handles=legend_elements, title="Sentiment")
for i, v in enumerate(all_top["size"]):
    axes[0].text(i, v + 3, str(v), ha="center", fontsize=7)

# Plot 2 — Sentiment breakdown for known communities (sorted by distrust)
if len(known) > 0:
    x = range(len(known))
    axes[1].bar(x, known["negative_pct"],
                color="#F44336", edgecolor="white", alpha=0.85, label="Negative %")
    axes[1].bar(x, known["positive_pct"],
                color="#4CAF50", edgecolor="white", alpha=0.85,
                bottom=known["negative_pct"].values, label="Positive %")
    axes[1].set_xticks(range(len(known)))
    axes[1].set_xticklabels([f"C{cid}" for cid in known["community_id"]], rotation=45, fontsize=8)
    axes[1].set_title("Sentiment Breakdown — Known Communities\n(sorted by distrust signal)")
    axes[1].set_xlabel("Community")
    axes[1].set_ylabel("Percentage of Users")
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5,
                 "Rerun text_analysis.ipynb\non combined_comments.csv\nto populate sentiment data",
                 ha="center", va="center", transform=axes[1].transAxes,
                 fontsize=11, color="gray")
    axes[1].set_title("Sentiment Breakdown per Community")
    axes[1].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "community_profiles.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved community_profiles.png")

## 9. Network Visualisation

Top 150 users by PageRank are visualised. Node colour = community. Node size = PageRank score. Edge thickness = interaction weight.

In [ ]:
top_nodes = centrality_df.head(150)["user"].tolist()
G_sub     = G.subgraph(top_nodes).copy()

unique_communities = list(set(partition.get(n, -1) for n in G_sub.nodes()))
cmap = plt.colormaps.get_cmap("tab20").resampled(len(unique_communities))
comm_color_map = {c: mcolors.to_hex(cmap(i)) for i, c in enumerate(unique_communities)}
node_colors  = [comm_color_map.get(partition.get(n, -1), "#9E9E9E") for n in G_sub.nodes()]

pr_vals      = [pagerank.get(n, 0) for n in G_sub.nodes()]
max_pr       = max(pr_vals) if pr_vals else 1
node_sizes   = [300 + (v / max_pr) * 2000 for v in pr_vals]

edge_weights = [G_sub[u][v].get("weight", 1) for u, v in G_sub.edges()]
max_w        = max(edge_weights) if edge_weights else 1
edge_widths  = [0.3 + (w / max_w) * 2 for w in edge_weights]

fig, ax = plt.subplots(figsize=(16, 14))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#0d1117")

pos = nx.spring_layout(G_sub, k=0.8, seed=42, iterations=50)

nx.draw_networkx_edges(G_sub, pos, ax=ax,
    edge_color="#444444", width=edge_widths, alpha=0.5,
    arrows=True, arrowsize=8, connectionstyle="arc3,rad=0.1")

nx.draw_networkx_nodes(G_sub, pos, ax=ax,
    node_color=node_colors, node_size=node_sizes, alpha=0.9)

top20_labels = {n: n[:20] for n in centrality_df.head(20)["user"].tolist() if n in G_sub.nodes()}
nx.draw_networkx_labels(G_sub, pos, labels=top20_labels, ax=ax,
    font_size=7, font_color="white")

ax.set_title("AI Coding Assistant Discussion Network\nTop 150 Users by PageRank (colour = community)",
             fontsize=14, fontweight="bold", color="white", pad=15)
ax.axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "network_graph.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("Saved network_graph.png")

## 10. Synthesis — Network Position × Sentiment × Topic

This directly answers the research question: *How do influential users and communities shape concerns around trust and distrust in AI coding assistants?*

- Do high-PageRank users lean positive or negative?
- Which concern themes attract the most engagement?
- Is there a relationship between network centrality and sentiment?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Network Position × Sentiment × Topic — Key Findings", fontsize=13, fontweight="bold")

# Plot 1 — Avg PageRank by sentiment
pr_by_sentiment = centrality_df.groupby("sentiment_label")["pagerank"].mean().sort_values(ascending=False)
bar_colors = [SENTIMENT_COLORS.get(s, "#9E9E9E") for s in pr_by_sentiment.index]
axes[0].bar(pr_by_sentiment.index, pr_by_sentiment.values, color=bar_colors, edgecolor="white")
axes[0].set_title("Avg PageRank by Sentiment\n(do negative users have more influence?)")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Average PageRank")
for i, v in enumerate(pr_by_sentiment.values):
    axes[0].text(i, v + 0.000001, f"{v:.6f}", ha="center", fontsize=8)

# Plot 2 — Avg PageRank by topic (exclude unknown)
pr_by_topic = centrality_df[centrality_df["dominant_topic"] != "unknown"].groupby("dominant_topic")["pagerank"].mean().sort_values(ascending=False)
bar_colors2 = [TOPIC_COLORS.get(t, "#9E9E9E") for t in pr_by_topic.index]
axes[1].bar(pr_by_topic.index, pr_by_topic.values, color=bar_colors2, edgecolor="white")
axes[1].set_title("Avg PageRank by Topic\n(which themes attract most engagement?)")
axes[1].set_xlabel("Topic")
axes[1].set_ylabel("Average PageRank")
axes[1].tick_params(axis="x", rotation=30)

# Plot 3 — Scatter: PageRank vs sentiment score (top 200 known users)
plot_df = centrality_df[centrality_df["sentiment_label"] != "unknown"].head(200)
scatter_colors = [SENTIMENT_COLORS.get(s, "#9E9E9E") for s in plot_df["sentiment_label"]]
axes[2].scatter(plot_df["avg_sentiment"], plot_df["pagerank"],
                c=scatter_colors, alpha=0.6, s=40, edgecolors="none")
axes[2].axvline(x=0, color="grey", linewidth=0.8, linestyle="--", alpha=0.7)
axes[2].set_title("PageRank vs Sentiment Score\n(top 200 users)")
axes[2].set_xlabel("Avg Sentiment Score")
axes[2].set_ylabel("PageRank")
legend_elements = [Patch(facecolor=v, label=k) for k, v in SENTIMENT_COLORS.items() if k != "unknown"]
axes[2].legend(handles=legend_elements, title="Sentiment", fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "synthesis_network_nlp.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved synthesis_network_nlp.png")

## 11. Summary

In [ ]:
print("=" * 55)
print("Network Analysis Complete")
print("=" * 55)
print(f"  Directed reply graph nodes:     {G.number_of_nodes():,}")
print(f"  Directed reply graph edges:     {G.number_of_edges():,}")
print(f"  Directed graph density:         {nx.density(G):.6f}")
print(f"  Connected components:           {len(components):,}")
print(f"  Largest component:              {len(largest_cc):,} nodes")
print(f"  Community graph nodes:          {G_community.number_of_nodes():,}")
print(f"  Community graph edges:          {G_community.number_of_edges():,}")
print(f"  Communities detected:           {num_communities:,}")
print(f"  Modularity:                     {modularity:.4f}")
print()
print("  Top 5 influential users (PageRank):")
for _, row in centrality_df.head(5).iterrows():
    print(f"    {row['user'][:35]:35s} PR: {row['pagerank']:.6f} | {row['sentiment_label']} | {row['dominant_topic']}")
print("=" * 55)
print()
print("Outputs saved to data/processed/network/:")
for f in sorted(OUTPUT_DIR.glob("*")):
    size = f.stat().st_size / 1024
    print(f"  {f.name} ({size:.0f} KB)")


## 12. Outputs

| File | Description |
|---|---|
| `centrality_scores.csv` | PageRank, in-degree, betweenness + sentiment/topic per user from the directed reply graph |
| `community_profiles.csv` | Community size, profiled users, dominant sentiment, and dominant topic from reply + co-participation edges |
| `degree_distribution.png` | In/out degree distribution for the directed reply graph |
| `top_users_centrality.png` | Top 20 users by PageRank coloured by sentiment and topic |
| `community_profiles.png` | Community sizes and sentiment breakdown |
| `network_graph.png` | Directed reply network visualisation — top 150 users, coloured by Louvain community |
| `synthesis_network_nlp.png` | PageRank vs sentiment/topic cross-analysis |

### Notes
PageRank and in-degree use direct reply edges only. GitHub issue comments are not direct replies, so GitHub contributes to community detection through projected user-user co-participation edges created from user-issue participation. This keeps issue IDs out of the user graph while preserving GitHub participation structure.
